In [1]:
import numpy as np
from numpy.typing import NDArray
import pandas as pd

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBRegressor
import lightgbm as lgb

import os
import sys
root_path="/home/iaw/DATA2/AAReact/src"
sys.path.append(root_path)
from util.RegressMetrics import r2_score, mse_score, mae_score, rmse_score
from util.train_tools import build_model, search_parms, split_data, load_data, eval_dataset_split
from tool.analysis import draw_pred_result

import seaborn as sns
from matplotlib import pyplot as plt

from typing import List, Tuple
from rich.table import Table
from rich import print as rp
from rich.progress import track
import shap
import pickle

from joblib import dump, load

In [5]:
seed = 1
test_size = 0.2
prefix = "/home/iaw/DATA2/AAReact/DataSet/Data_All/3_data_for_train/rdkit_xtb_acsf/train_test"
data_x, data_y, x_label, data_class, data_name = load_data(
    "{}_data_x.npy".format(prefix), 
    "{}_data_y.npy".format(prefix), 
    "{}_x_label.pkl".format(prefix), 
    "{}_data_class.pkl".format(prefix), 
    "{}_data_name.pkl".format(prefix)
)

# 超参之后的参数
xgb_best_params = {
    "colsample_bytree": 0.8,
    "learning_rate": 0.1,
    "max_depth": 10,
    "min_child_weight": 5,
    "n_estimators": 50,
    "reg_alpha": 0.5,
    "reg_lambda": 0.5,
    "subsample": 0.8
}

xgb_eval_dataset_split_result = eval_dataset_split(
                     seed_s = [1, 2, 3, 4]
                   , test_size_s = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
                   , parms = xgb_best_params
                   , model_name = "xgb"
                   , data_x = data_x, data_y = data_y, data_class = data_class
                   , eval_func = r2_score) 




In [29]:
data_x.shape

(444, 392)

In [6]:
# ====================== 全局 SCI 样式（和你之前的图表完全统一）======================
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'font.size': 11,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'axes.linewidth': 1.2,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
    'figure.dpi': 300,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'savefig.format': 'pdf'
})

# ====================== 模型敏感性绘图（优化版）======================
test_size_s = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

plt.figure(figsize=(8, 5), dpi=300)

# --- 训练集曲线（统一论文配色 + 高级样式）---
plt.errorbar(
    test_size_s,
    xgb_eval_dataset_split_result[0],
    yerr=xgb_eval_dataset_split_result[1],
    label="Train XGB",
    fmt="-o",
    capsize=5,
    capthick=1.5,
    elinewidth=1.5,
    linewidth=2.5,
    markersize=7,
    markerfacecolor="#4C72B0",   # 统一训练集蓝色
    markeredgecolor="white",
    markeredgewidth=1.2,
    color="#4C72B0"
)

# --- 测试集曲线（统一论文配色）---
plt.errorbar(
    test_size_s,
    xgb_eval_dataset_split_result[2],
    yerr=xgb_eval_dataset_split_result[3],
    label="Test XGB",
    fmt="-s",
    capsize=5,
    capthick=1.5,
    elinewidth=1.5,
    linewidth=2.5,
    markersize=7,
    markerfacecolor="#DD8452",   # 统一测试集橙色
    markeredgecolor="white",
    markeredgewidth=1.2,
    color="#DD8452"
)

# 基准虚线
plt.plot([0, 1], [0.5, 0.5], linestyle='--', c="#000000", linewidth=1.2, label='Baseline ($R^2=0.5$)')

# --- 标签、图例、网格（SCI 标准）---
plt.xlabel("Test Size", fontweight='bold')
plt.ylabel("$R^2$", fontweight='bold')
plt.xlim(0, 1)
plt.ylim(0, 1)

plt.grid(True, axis='y', linestyle='--', alpha=0.3, linewidth=0.8)  # 柔和网格
plt.legend(frameon=True, edgecolor='black', fancybox=False)        # 专业图例

plt.tight_layout()

# 保存论文级图片
plt.savefig('model_sensitivity_test_size.png', dpi=600)
plt.close()


In [7]:
model = load("/home/iaw/DATA2/AAReact/train/output/pt/xgb_rdkit_xtb_acsf_seed_0-1_test_0-2.pkl")
X_train, X_test, y_train, y_test,  class_train, class_test, name_train, name_test = train_test_split(
    data_x,        
    data_y,
    data_class,
    data_name,
    test_size=test_size,
    random_state=seed, 
)

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

In [8]:
pred_error_sign = list(np.where(train_pred*y_train < 0)[0])
print("The num of sign-error is {}".format(len(pred_error_sign)))

The num of sign-error is 16


In [24]:

# ====================== 全局 SCI 样式（完全统一）======================
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'font.size': 11,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'axes.linewidth': 1.2,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
    'figure.dpi': 300,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'savefig.format': 'pdf'
})

# ====================== 你的数据（直接替换）======================
y_true_s = (y_train, y_test)
y_pred_s = (train_pred, test_pred)

r2_s = (
    r2_score(y_pred_s[0], y_true_s[0]),
    r2_score(y_pred_s[1], y_true_s[1])
)

# ====================== 绘图 ======================
fig, ax = plt.subplots(figsize=(7, 7), dpi=300)

# 配色
color_train = "#4C72B0"
color_test  = "#DD8452"
color_diag  = "#000000"

# 绘制散点
ax.scatter(y_true_s[0], y_pred_s[0], 
           c=color_train, alpha=0.6, s=35, label=f"Train ($R^2$ = {r2_s[0]:.4f})",
           edgecolors='white', linewidth=0.5)
ax.scatter(y_true_s[1], y_pred_s[1], 
           c=color_test, alpha=0.7, s=35, label=f"Test ($R^2$ = {r2_s[1]:.4f})",
           edgecolors='white', linewidth=0.5)

# 范围
lims = [-1, 1]

# 对角线 y=x
x_line = np.linspace(lims[0], lims[1], 100)
ax.plot(x_line, x_line, '--', color=color_diag, linewidth=2)

# ====================== ✅ 你要的区域：左上 + 右下 红色填充 ======================
# 左上：x from -1 to 0, y from 0 to 1
ax.axvspan(xmin=-1, xmax=0, ymin=0.5, ymax=1, color='red', alpha=0.15, zorder=0)

# 右下：x from 0 to 1, y from -1 to 0
ax.axvspan(xmin=0, xmax=1, ymin=0, ymax=0.5, color='red', alpha=0.15, zorder=0)

# ====================== 样式 ======================
ax.set_xlabel('True Value', fontweight='bold')
ax.set_ylabel('Predicted Value', fontweight='bold')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect('equal', adjustable='box')

ax.grid(True, linestyle='--', alpha=0.3, lw=0.8)
ax.legend(frameon=True, edgecolor='black', fancybox=False, loc='upper left')

plt.tight_layout()
plt.savefig('regression_parity_plot.png', dpi=600)
plt.close()

In [27]:
x_label.keys()

dict_keys(['label1_rdkit', 'label2_xtb', 'label3_acsf'])

In [28]:


# ====================== 全局 SCI 样式（和你所有图表完全统一）======================
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'axes.linewidth': 1.2,
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
    'figure.dpi': 300,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'savefig.format': 'pdf'
})

# ====================== SHAP 计算 ======================
xgb_explainer = shap.TreeExplainer(model)
train_shap_values = xgb_explainer.shap_values(X_train)
labels = x_label["label1_rdkit"] + x_label["label2_xtb"] + x_label["label3_acsf"]

# ====================== 绘图：SHAP 蜂群图 + 重要度柱状图 ======================
fig, ax1 = plt.subplots(figsize=(10, 8), dpi=300)

# 绘制蜂群图 (dot)
shap.summary_plot(
    train_shap_values, 
    X_train, 
    feature_names=labels, 
    plot_type="dot", 
    show=False,
    plot_size=None  # 关键：让shap使用我们的fig/ax
)

# 创建上X轴，绘制条形图 (bar)
ax2 = ax1.twiny()
shap.summary_plot(
    train_shap_values, 
    X_train, 
    feature_names=labels, 
    plot_type="bar", 
    show=False,
    plot_size=None
)

# 让 bar 半透明，不遮挡蜂群图
for patch in ax2.patches:
    patch.set_alpha(0.2)

# 分界线
ax2.axhline(y=20, color='black', linestyle='-', linewidth=1.0)

# 轴标签（统一论文风格）
ax1.set_xlabel('SHAP Value (Bee Swarm Plot)', fontweight='bold', labelpad=10)
ax2.set_xlabel('Mean |SHAP Value| (Feature Importance)', fontweight='bold', labelpad=10)
ax1.set_ylabel('Features', fontweight='bold')

# 上轴显示在顶部
ax2.xaxis.set_label_position('top')
ax2.xaxis.tick_top()

# 布局与保存
plt.tight_layout()
plt.savefig('shap_summary_beeswarm_bar.png', dpi=600)
plt.close()
